# Pure-Python asyncio patterns

These cells exercise the same patterns without the HTTP layer so you can step through them.

In [ ]:
import asyncio, time, random

async def call(i, ms=100):
    await asyncio.sleep((ms + random.randint(0,50))/1000)
    return i

async def run():
    t = time.perf_counter()
    res = await asyncio.gather(*(call(i) for i in range(20)))
    print('gather 20:', int((time.perf_counter()-t)*1000), 'ms')

    sem = asyncio.Semaphore(5)
    async def guarded(i):
        async with sem:
            return await call(i)
    t = time.perf_counter()
    res = await asyncio.gather(*(guarded(i) for i in range(20)))
    print('semaphore(5) 20:', int((time.perf_counter()-t)*1000), 'ms')

    try:
        async with asyncio.timeout(0.05):
            await call(0, 200)
    except asyncio.TimeoutError:
        print('timeout fired (expected)')

await run()

In [ ]:
# TaskGroup cancels siblings when one fails
async def boom():
    await asyncio.sleep(0.1); raise RuntimeError('nope')
async def long():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print('long cancelled cleanly')
        raise

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(long())
        tg.create_task(boom())
except* RuntimeError as eg:
    print('caught', eg.exceptions)